# バックテスト実験サンプル Notebook

このノートブックは、bt-log-vis-toolを使った実験データ保存のサンプルです。

In [ ]:
import numpy as np
import pandas as pd
from bt_log_vis_tool import ExperimentSaver

## 1. 実験設定

In [ ]:
# 実験設定
BASE_DIR = "~/backtest_experiments"
EXP_NAME = "momentum_strategy"
RUN_NAME = "run_001"

# Saver初期化
# non_target_columns_ticker: 銘柄データ用の非ターゲットカラム
# non_target_columns_strategy: 戦略データ用の非ターゲットカラム
# 例: 銘柄データにはrandom_seedがあるが、戦略データはシードアンサンブル後なので不要
#     non_target_columns_ticker=["run_id", "random_seed"]
#     non_target_columns_strategy=["run_id"]
# デフォルトでは両方とも["split", "epoch"]のみ
saver = ExperimentSaver(BASE_DIR, EXP_NAME, RUN_NAME)
print(f"保存先: {saver.run_dir}")

## 2. バックテスト実行（サンプルデータ生成）

In [ ]:
# 日付範囲
dates = pd.date_range("2023-01-01", "2023-12-31", freq="D")
n_dates = len(dates)
n_epochs = 10

print(f"日数: {n_dates}")
print(f"エポック数: {n_epochs}")

In [ ]:
# 戦略毎PnLデータの作成
pnl_strategy_data = {"split": [], "run_id": [], "epoch": []}

for epoch in range(n_epochs):
    for split in ["train", "val", "test"]:
        n_dates_split = n_dates // 3
        pnl_strategy_data["split"].extend([split] * n_dates_split)
        pnl_strategy_data["run_id"].extend([f"epoch_{epoch}"] * n_dates_split)
        pnl_strategy_data["epoch"].extend([epoch] * n_dates_split)

pnl_strategy_df = pd.DataFrame(pnl_strategy_data, index=dates[: len(pnl_strategy_data["split"])])

# 戦略列の追加
for strategy in ["strategy_long", "strategy_short", "strategy_longshort"]:
    pnl_strategy_df[strategy] = np.random.randn(len(pnl_strategy_df)) * 0.01

print("PnL DataFrame:")
print(pnl_strategy_df.head())

In [ ]:
# ポジションデータの作成
position_strategy_df = pnl_strategy_df.copy()
for strategy in ["strategy_long", "strategy_short", "strategy_longshort"]:
    position_strategy_df[strategy] = np.random.choice([-1, 0, 1], size=len(position_strategy_df))

print("Position DataFrame:")
print(position_strategy_df.head())

In [ ]:
# 統計メトリクスデータの作成
stats_data = []
for epoch in range(n_epochs):
    for split in ["train", "val", "test"]:
        stats_data.append(
            {
                "epoch": epoch,
                "split": split,
                "run_id": f"epoch_{epoch}",
                "annual_return": np.random.uniform(0.05, 0.20),
                "annual_risk": np.random.uniform(0.10, 0.25),
                "sharpe_ratio": np.random.uniform(0.5, 2.0),
                "max_drawdown": -np.random.uniform(0.05, 0.15),
            }
        )

stats_df = pd.DataFrame(stats_data).set_index("epoch")

print("Stats DataFrame:")
print(stats_df.head(10))

## 3. パラメータとコードの設定

In [ ]:
# ハイパーパラメータ
params = {
    "model": {
        "type": "neural_network",
        "layers": [128, 64, 32],
        "activation": "relu",
        "dropout": 0.2,
    },
    "training": {
        "epochs": n_epochs,
        "batch_size": 256,
        "learning_rate": 0.001,
        "optimizer": "adam",
    },
    "strategy": {
        "long_threshold": 0.6,
        "short_threshold": -0.6,
        "rebalance_freq": "daily",
    },
}

print("Parameters:")
print(params)

In [ ]:
# 実験コード（文字列として保存）
code = '''
# バックテスト実験コード
import pandas as pd
import numpy as np

# データ読み込み
data = load_data()

# モデル学習
model = train_model(data)

# 予測
predictions = model.predict(data)

# 戦略実行
pnl = execute_strategy(predictions)
'''

## 4. データ保存

In [ ]:
# 全データを一括保存
saver.save_all(
    pnl_strategy=pnl_strategy_df,
    position_strategy=position_strategy_df,
    stats_metrics=stats_df,
    params=params,
    code=code,
    code_filename="experiment.py",
)

print("\n保存完了!")
print(f"保存先: {saver.run_dir}")

## 5. ダッシュボードの起動

ターミナルで以下のコマンドを実行して、可視化ダッシュボードを起動します:

```bash
streamlit run bt_log_vis_tool/app.py
```

または個別保存する場合:

In [ ]:
# 個別保存の例
# saver.save_pnl_strategy(pnl_strategy_df)
# saver.save_position_strategy(position_strategy_df)
# saver.save_stats_metrics(stats_df)
# saver.save_params(params)
# saver.save_code(code, "experiment.py")